# Stage 1: Price, Momentum, Fundamentals

Build core traditional factors and save all reusable outputs to parquet.

In [1]:
import io
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'intermediate'
DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = (pd.Timestamp.today().normalize() - pd.DateOffset(years=15)).date()
END_DATE = pd.Timestamp.today().normalize().date()
WARMUP_YEARS = 1
FETCH_START = (pd.Timestamp.today().normalize() - pd.DateOffset(years=15 + WARMUP_YEARS)).date()

print('ROOT:', ROOT)
print('Window:', FETCH_START, '->', END_DATE)

ROOT: C:\Users\aidan\Documents\CS229Project\cs229project
Window: 2010-03-08 -> 2026-03-08


In [2]:
def get_sp500_tickers():
    try:
        tickers = yf.tickers_sp500()
        return [t.replace('.', '-') for t in tickers]
    except Exception:
        import requests
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        df = pd.read_html(io.StringIO(resp.text))[0]
        return [t.replace('.', '-') for t in df['Symbol'].tolist()]

def download_close_prices(tickers, start, end):
    data = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        interval='1d',
        group_by='ticker',
        progress=False,
        auto_adjust=True,
        threads=True
    )
    if isinstance(data.columns, pd.MultiIndex):
        close = pd.DataFrame({t: data[t]['Close'] for t in tickers if t in data.columns.levels[0]})
    else:
        close = data['Close'].to_frame(tickers[0])
    return close.sort_index()

def momentum_12m_ex1m(close_df, lb_days=252, skip_days=21):
    return (close_df.shift(skip_days) / close_df.shift(lb_days)) - 1.0

In [3]:
tickers = get_sp500_tickers()
close = download_close_prices(tickers, FETCH_START, END_DATE)
momentum = momentum_12m_ex1m(close)

print('Universe size:', len(tickers))
print('Close shape:', close.shape)

pd.Series(tickers, name='ticker').to_csv(DATA_DIR / 'sp500_tickers.csv', index=False)
close.to_parquet(DATA_DIR / 'close_prices.parquet')
momentum.to_parquet(DATA_DIR / 'momentum_12m_ex1m.parquet')


1 Failed download:
['CB']: TypeError("'NoneType' object is not subscriptable")


Universe size: 503
Close shape: (4025, 503)


In [4]:
def get_fundamentals(t):
    tk = yf.Ticker(t)
    try:
        info = tk.get_info()
    except Exception:
        info = {}

    def get_or_nan(k):
        v = info.get(k, np.nan)
        return np.nan if v is None else v

    metrics = {
        'roe': get_or_nan('returnOnEquity'),
        'roa': get_or_nan('returnOnAssets'),
        'profit_margins': get_or_nan('profitMargins'),
        'gross_margins': get_or_nan('grossMargins'),
        'debt_to_equity': get_or_nan('debtToEquity'),
        'trailing_pe': get_or_nan('trailingPE'),
        'ev_to_ebitda': get_or_nan('enterpriseToEbitda'),
        'eps_ttm': get_or_nan('epsTrailingTwelveMonths'),
        'shares_out': get_or_nan('sharesOutstanding'),
        'price_to_book': get_or_nan('priceToBook'),
        'book_value_ps': get_or_nan('bookValue'),
    }

    equity = np.nan
    try:
        bs = tk.quarterly_balance_sheet
        for key in [
            'StockholdersEquity',
            'Total Stockholder Equity',
            'Total Equity Gross Minority Interest',
            'Total shareholders equity',
            'Total Stockholders Equity'
        ]:
            if key in bs.index:
                val = pd.to_numeric(bs.loc[key].iloc[0], errors='coerce')
                if not pd.isna(val):
                    equity = val
                    break
    except Exception:
        pass
    metrics['equity_latest'] = equity
    return metrics

fund_list = []
for t in tqdm(tickers, desc='Fetching fundamentals'):
    try:
        fund_list.append((t, get_fundamentals(t)))
    except Exception:
        fund_list.append((t, {
            'roe': np.nan, 'roa': np.nan, 'profit_margins': np.nan, 'gross_margins': np.nan,
            'debt_to_equity': np.nan, 'trailing_pe': np.nan, 'ev_to_ebitda': np.nan,
            'eps_ttm': np.nan, 'shares_out': np.nan, 'price_to_book': np.nan,
            'book_value_ps': np.nan, 'equity_latest': np.nan
        }))

fund = pd.DataFrame({t: m for t, m in fund_list}).T
fund.to_parquet(DATA_DIR / 'fundamentals_snapshot.parquet')
print('Fundamentals shape:', fund.shape)

Fetching fundamentals: 100%|██████████| 503/503 [03:09<00:00,  2.66it/s]

Fundamentals shape: (503, 12)


In [5]:
eps = fund['eps_ttm']
pe = fund['trailing_pe']
shares_out = fund['shares_out']
equity = fund['equity_latest']
pb = fund['price_to_book']
bvps = fund['book_value_ps']
ev_ebitda = fund['ev_to_ebitda']

def wide_to_long(df, colname):
    out = df.stack().to_frame(colname)
    out.index.set_names(['date', 'ticker'], inplace=True)
    return out

def cs_zscore(s):
    valid = s.dropna()
    if len(valid) < 5:
        return pd.Series(np.nan, index=s.index)
    sd = valid.std(ddof=0)
    if sd == 0 or pd.isna(sd):
        return pd.Series(np.nan, index=s.index)
    return (s - valid.mean()) / sd

def ep_col(col):
    t = col.name
    base = eps.get(t, np.nan)
    if not pd.isna(base) and base != 0:
        return base / col
    alt_pe = pe.get(t, np.nan)
    if not pd.isna(alt_pe) and alt_pe != 0:
        return pd.Series(1.0 / alt_pe, index=col.index)
    return pd.Series(np.nan, index=col.index)

def bm_series(col):
    t = col.name
    eq = equity.get(t, np.nan)
    sh = shares_out.get(t, np.nan)
    if not pd.isna(eq) and not pd.isna(sh) and sh != 0:
        return eq / (sh * col)
    if not pd.isna(pb.get(t, np.nan)) and pb.get(t) != 0:
        return pd.Series(1.0 / pb.get(t), index=col.index)
    if not pd.isna(bvps.get(t, np.nan)):
        return bvps.get(t) / col
    return pd.Series(np.nan, index=col.index)

ep_daily = close.apply(ep_col)
bm_daily = close.apply(bm_series)
ev_daily = pd.DataFrame(index=close.index, columns=close.columns, data=np.nan)
for t in close.columns:
    ev_daily[t] = ev_ebitda.get(t, np.nan)

value_raw = (
    wide_to_long(ep_daily, 'earnings_yield')
    .join(wide_to_long(bm_daily, 'book_to_market'), how='outer')
    .join(wide_to_long(ev_daily, 'ev_to_ebitda'), how='outer')
)
value_z = value_raw.groupby(level='date').transform(cs_zscore)
value_z['ev_to_ebitda'] = -value_z['ev_to_ebitda']
value_z['value_composite'] = value_z[['earnings_yield', 'book_to_market', 'ev_to_ebitda']].mean(axis=1, skipna=True)

quality_base = fund[['roe', 'roa', 'profit_margins', 'gross_margins', 'debt_to_equity']].copy()
q_z = quality_base.apply(lambda c: (c - c.mean()) / c.std(ddof=0))
q_z['debt_to_equity'] = -q_z['debt_to_equity']
q_z['quality_composite'] = q_z[['roe', 'roa', 'profit_margins', 'gross_margins', 'debt_to_equity']].mean(axis=1, skipna=True)

q_daily = pd.DataFrame(index=close.index, columns=close.columns, data=np.nan)
for t in close.columns:
    q_daily[t] = q_z.loc[t, 'quality_composite'] if t in q_z.index else np.nan

momentum_long = wide_to_long(momentum, 'momentum')
quality_long = wide_to_long(q_daily, 'quality_composite')

factors = (
    momentum_long[['momentum']]
    .join(value_z[['value_composite']], how='outer')
    .join(quality_long[['quality_composite']], how='outer')
).sort_index()

start_ts = pd.Timestamp(START_DATE)
end_ts = pd.Timestamp(END_DATE)
factors = factors.loc[(slice(start_ts, end_ts), slice(None)), :]

next_ret = close.pct_change(fill_method=None).shift(-1)
next_ret_long = next_ret.stack().to_frame('ret_t1')
next_ret_long.index.set_names(['date', 'ticker'], inplace=True)

factors.to_parquet(DATA_DIR / 'factors_traditional.parquet')
next_ret_long.to_parquet(DATA_DIR / 'target_next_ret.parquet')
print('Saved factors_traditional.parquet and target_next_ret.parquet')

Saved factors_traditional.parquet and target_next_ret.parquet
